# Pangolin — splice-site prediction & variant scoring

![Pangolin — splice-site prediction & variant scoring](https://proto-bio.github.io/proto-assets/images/tool/pangolin/hero.png)

Pangolin is a SpliceAI-lineage deep-learning model that predicts tissue-specific splice-site probability from raw DNA and scores the splicing effect (gain/loss) of variants. This notebook demonstrates both registered tools: `run_pangolin_predict` for per-position splice-probability prediction and `run_pangolin_score_variants` for variant splice-effect scoring.

- Paper: https://doi.org/10.1186/s13059-022-02664-4
- Repository: https://github.com/tkzeng/Pangolin

In [1]:
# Public API for both Pangolin tools, plus the notebook docs helper.
from proto_tools.tools.rna_splicing.pangolin import (
    run_pangolin_predict,
    PangolinPredictInput,
    PangolinPredictConfig,
    run_pangolin_score_variants,
    PangolinScoreVariantsInput,
    PangolinScoreVariantsConfig,
    PangolinVariant,
)
from proto_tools.utils.notebook_docs import display_api_reference

In [2]:
# Input schema for splice-site prediction: the DNA sequences to score.
display_api_reference("pangolin-predict", "input", "run_pangolin_predict")

**Input** — `PangolinPredictInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | DNA sequence(s) >= 10001 bp; predictions cover the central (len - 10000) positions |

In [3]:
# Config schema: which tissues to ensemble and which device to run on.
display_api_reference("pangolin-predict", "config", "run_pangolin_predict")

**Config** — `PangolinPredictConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>tissues</code> | <code>list[Literal['HEART', 'LIVER', 'BRAIN', 'TESTIS']]</code> | <code>['HEART', 'LIVER', 'BRAIN', 'TESTIS']</code> | Tissues whose splice predictions are ensembled (HEART, LIVER, BRAIN, TESTIS) |

In [4]:
# Output schema: per-sequence predictions with per-position, per-tissue scores.
display_api_reference("pangolin-predict", "output", "run_pangolin_predict")

**Output** — `PangolinPredictOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[PangolinPrediction]</code> | required | Per-sequence Pangolin predictions (1:1 with input sequences) |

**`PangolinPrediction`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>scores</code> | <code>list[list[float]]</code> | required | Per-position splice-site probability scores, shape [position][tissue] |
| <code>tissues</code> | <code>list[Literal['HEART', 'LIVER', 'BRAIN', 'TESTIS']]</code> | required | Tissue order of the score columns |
| <code>output_start</code> | <code>int</code> | required | Input-sequence index of the first scored position (= PANGOLIN_FLANK) |

## Splice-site prediction

Pangolin needs at least 10,001 bp of input: it uses 5,000 bp of flanking context on each side and scores the central `len(sequence) - 10000` positions. We build a reproducible random 10,100 bp sequence so the example runs without any external genome file.

In [5]:
from pathlib import Path

# Real HBB (hemoglobin beta) genomic window. Pangolin scores the central positions
# (5,000 bp of flank on each side), which span the HBB splice sites.
hbb = Path("hbb_genomic.fasta").read_text()
seq = "".join(line for line in hbb.splitlines() if not line.startswith(">"))

result = run_pangolin_predict(
    PangolinPredictInput(sequences=[seq]),
    PangolinPredictConfig(device="cpu"),
)

prediction = result.results[0]
print(f"success: {result.success}")
print(f"scored positions: {len(prediction.scores)}")
print(f"tissues: {prediction.tissues}")
# Strongest splice signal across the scored window (max |score| over tissues).
import numpy as np
arr = np.abs(np.array(prediction.scores))
best = int(arr.max(axis=1).argmax())
print(f"max splice score {arr[best].max():.3f} at scored index {best}")

Running run_pangolin_predict [00:00]

success: True
scored positions: 106
tissues: ['HEART', 'LIVER', 'BRAIN', 'TESTIS']
max splice score 0.766 at scored index 32


In [6]:
# Advanced: restrict the ensemble to a subset of tissues. Each score row then
# has one column per requested tissue, in the order given.
subset = run_pangolin_predict(
    PangolinPredictInput(sequences=[seq]),
    PangolinPredictConfig(device="cpu", tissues=["BRAIN", "HEART"]),
)

subset_prediction = subset.results[0]
print(f"tissues: {subset_prediction.tissues}")
print(f"score shape: {len(subset_prediction.scores)} positions x "
      f"{len(subset_prediction.scores[0])} tissues")

Running run_pangolin_predict [00:00]

tissues: ['BRAIN', 'HEART']
score shape: 106 positions x 2 tissues


## Variant splice-effect scoring

`run_pangolin_score_variants` compares the predicted splice-site probability between the reference and alternate sequence over a `± distance` window, reducing across the selected tissues to a per-position splice gain (largest increase) and loss (largest decrease). Each variant carries its own reference window — no genome FASTA is required.

In [7]:
# Place a single-nucleotide variant at the HBB splice site found above (position 5032),
# reusing the HBB window (Pangolin needs >= 5,000 bp of flank on each side).
pos = 5032
ref = seq[pos]
alt = "A" if ref != "A" else "G"

variant = PangolinVariant(
    sequence=seq,
    variant_position=pos,
    reference_bases=ref,
    alternate_bases=alt,
)

# distance=50 reports splice scores 50 bp on each side of the variant.
result = run_pangolin_score_variants(
    PangolinScoreVariantsInput(variants=[variant]),
    PangolinScoreVariantsConfig(device="cpu", distance=50),
)

effect = result.results[0]
print(f"success: {result.success}")
print(f"largest increase: {effect.increase_score:.4f} at {effect.increase_position} bp")
print(f"largest decrease: {effect.decrease_score:.4f} at {effect.decrease_position} bp")
print(f"metrics: {effect.metrics}")

Running run_pangolin_score_variants [00:00]

success: True
largest increase: 0.0476 at 0 bp
largest decrease: -0.0115 at 13 bp
metrics: primary_metric='max_gain' metric_type='PangolinVariantMetrics' max_gain=0.04757942631840706 max_loss=-0.01149823796004057


In [8]:
# Schemas for the variant-scoring tool: config (tissues, distance, device) and output.
display_api_reference("pangolin-score-variants", "config", "run_pangolin_score_variants")
display_api_reference("pangolin-score-variants", "output", "run_pangolin_score_variants")

**Config** — `PangolinScoreVariantsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>tissues</code> | <code>list[Literal['HEART', 'LIVER', 'BRAIN', 'TESTIS']]</code> | <code>['HEART', 'LIVER', 'BRAIN', 'TESTIS']</code> | Tissues whose splice predictions are ensembled (HEART, LIVER, BRAIN, TESTIS) |
| <code>distance</code> | <code>int</code> | <code>50</code> | bp on each side of the variant to report splice scores for |

**Output** — `PangolinScoreVariantsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[PangolinVariantEffect]</code> | required | Per-variant splice-effect scores (1:1 with input variants) |

**`PangolinVariantEffect`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>loss_scores</code> | <code>list[float]</code> | required | Per-position splice-loss scores over the reporting window |
| <code>gain_scores</code> | <code>list[float]</code> | required | Per-position splice-gain scores over the reporting window |
| <code>increase_position</code> | <code>int</code> | required | Position (bp, relative to variant) of the largest increase |
| <code>increase_score</code> | <code>float</code> | required | Largest splice-score increase |
| <code>decrease_position</code> | <code>int</code> | required | Position (bp, relative to variant) of the largest decrease |
| <code>decrease_score</code> | <code>float</code> | required | Largest splice-score decrease |
| <code>metrics</code> | <code>PangolinVariantMetrics</code> | required | Scalar splice-effect summary metrics |

**Metrics** — `PangolinVariantMetrics` (one set per `results` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>max_gain</code> **(primary)** | <code>float</code> | <code>[-1, 1]</code> |  |  |
| <code>max_loss</code> | <code>float</code> | <code>[-1, 1]</code> |  |  |

In [9]:
# Export the variant-effect results. The default format is JSON; CSV is also
# supported via file_format="csv" (one row per variant with gain/loss summaries).
from pathlib import Path
Path("example_output").mkdir(exist_ok=True)
result.export(name="pangolin_variants", export_path="./example_output")
print("Exported results to example_output/pangolin_variants.json (default JSON; pass file_format='csv' for CSV)")

Exported results to example_output/pangolin_variants.json (default JSON; pass file_format='csv' for CSV)
